<a href="https://colab.research.google.com/github/G0nkly/pytorch_sandbox/blob/main/gpts/nanoGPT/AK_GPT_DIY_RECAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Load the data
# Encoding / Decoding
# Train/Test Split | get_batch function

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-09-19 07:52:54--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  6.46MB/s    in 0.2s    

2026-09-19 07:52:54 (6.46 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
import torch
import torch.functional as F
import torch.nn as nn

In [4]:
with open("./input.txt") as file:
  text = file.read()

In [5]:
vocab = sorted(set([c for c in text.strip()]))
stoi = {v: k  for k,v in enumerate(vocab)}
itos = {k: v  for k,v in enumerate(vocab)}
encode = lambda chars: [stoi[char] for char in chars]
decode = lambda indices: "".join(itos[index] for index in indices)

In [34]:
###################
# HYPERPARAMETERS #
###################
vocab_size = len(vocab)
block_size = 8
batch_size = 8
embed_dim = 32
eval_iter = 1000
train_test_split = 0.8
device = "cuda" if torch.cuda.is_available() else "cpu"

In [35]:
data = torch.tensor(encode(text))

In [36]:
data[:10]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])

In [37]:
train_index = int(train_test_split * len(data))
train = data[:train_index]
test = data[train_index:]
indices = torch.randint(0, len(train) - block_size, (batch_size,))
indices

tensor([709478, 612939, 542860, 492922, 787640, 255833, 630734, 167020])

In [38]:
indices = torch.randint(0, len(train) - block_size,(1,8))
indices = indices.squeeze(0).tolist()
x = []
y = []
for index in indices:
  x.append(torch.stack([train[index + i] for i in range(0, block_size)]))
  y.append(torch.stack([train[index + i + 1] for i in range(0, block_size)]))

X = torch.stack(x)
Y = torch.stack(y)

X, Y


(tensor([[61, 46, 43, 56, 43,  1, 63, 53],
         [ 0, 21,  1, 49, 52, 53, 61,  1],
         [ 1, 54, 50, 39, 52, 58,  6,  0],
         [25, 43, 39, 57, 59, 56, 43, 42],
         [57, 43, 56, 60, 47, 41, 43,  6],
         [53,  1, 51, 53, 60, 43,  1, 63],
         [ 1, 39, 50, 53, 52, 43, 11,  0],
         [21, 21, 10,  0, 24, 53,  6,  1]]),
 tensor([[46, 43, 56, 43,  1, 63, 53, 59],
         [21,  1, 49, 52, 53, 61,  1, 52],
         [54, 50, 39, 52, 58,  6,  0, 35],
         [43, 39, 57, 59, 56, 43, 42,  1],
         [43, 56, 60, 47, 41, 43,  6,  1],
         [ 1, 51, 53, 60, 43,  1, 63, 53],
         [39, 50, 53, 52, 43, 11,  0, 35],
         [21, 10,  0, 24, 53,  6,  1, 39]]))

In [39]:
def get_batch(split = "train"):
  data = train if split == "train" else test
  indices = torch.randint(0, len(train) - block_size, (batch_size,))

  x = []
  y = []
  for index in indices:
    x.append(torch.stack([train[index + i] for i in range(0, block_size)]))
    y.append(torch.stack([train[index + i + 1] for i in range(0, block_size)]))

  X = torch.stack(x)
  Y = torch.stack(y)

  return X, Y

In [40]:
@torch.no_grad()
def evaluate_model(model):
  model.eval()
  losses = torch.zeros(eval_iter)
  for i in range(eval_iter):
    x_train, y_train = get_batch("test")
    pred = model(x_train)
    B, T, C = pred.shape
    pred = pred.view(B*T, C)
    y_train = y_train.view(B * T)
    loss = F.cross_entropy(pred, y_train)
    losses[i] = loss
  model.train()
  return losses.mean()

In [41]:
################
# BUILD MODELS #
################

In [42]:
class BigramModel(nn.Module):

  def __init__(self):
    super().__init__()
    self.embd = nn.Embedding(vocab_size, vocab_size)
  def forward(self, inputs, targets=None):
    x = self.embd(inputs)
    if targets is not None:
      B, T, C = x.shape
      targets = targets.view(B*T)
      loss = F.cross_entropy(x.view(B*T,C), targets)
      return x, loss
    return x, None

  def generate(self, start_token, n_tokens):
    for _ in range(n_tokens):
      input = start_token[-block_size:]                     # (B, T, C)
      logits, loss = self(input)                            # (B, T, C)
      logits = logits[:, -1, :]                             # (B, 1, C)
      probs = nn.functional.softmax(logits, dim=-1)         # (B, 1, C)
      idx = torch.multinomial(probs, num_samples=1)         # (B, 1)
      start_token = torch.cat((input, idx), dim=-1)         # (B, T+1)

    return start_token


In [43]:
emb = nn.Embedding(vocab_size, vocab_size)
preds = emb(torch.tensor(0).unsqueeze(0))
probs = nn.functional.softmax(preds, -1)
idx = torch.multinomial(probs, 1)
preds.shape, probs.shape

(torch.Size([1, 65]), torch.Size([1, 65]))

In [44]:
model = BigramModel()
start_token = torch.tensor(encode(" "))
token_ids = model.generate(start_token.unsqueeze(0), 100).squeeze().tolist()
decode(token_ids)

" D\nl3w\n:v?m3dfm-ITJ'EVSFw -d,xe$QrBvX-idfiFh$aNj:MNnvrPT3 !hiE3DJ;rzDHc?eSXwaWfNsK3q\nqqNZ'rXsvnXsgXnj"

In [52]:
class TheGodamnTransformer(nn.Module):

  def __init__(self):
    super().__init__()
    self.embed = nn.Embedding(vocab_size, embed_dim)
    self.positional_encoding = nn.Embedding(block_size, embed_dim)

  def forward(self, x):
    x = self.embed(x) # B, T, C
    x = x + self.positional_encoding(torch.arange(0, block_size, device=device))


    return x

In [53]:
transformer = TheGodamnTransformer()
input = torch.tensor(1).unsqueeze(0)
output = transformer(input)
output.shape

torch.Size([8, 32])